# Data Perparation

We will use `HFTBacktest` as both the backtesting framework and the reinforcement learning environment.
Therefore, it is essential to preprocess and convert our market data into the format required by `HFTBacktest`.
Please refer to the official documentation for data format specifications: [HFTBacktest Data Guide](https://hftbacktest.readthedocs.io/en/latest/data.html)


## Load dependencies


In [1]:
import json
import csv
import numpy as np
import os
from hftbacktest import event_dtype
from hftbacktest import EVENT_ARRAY, BUY_EVENT, SELL_EVENT, DEPTH_EVENT, TRADE_EVENT, EXCH_EVENT, LOCAL_EVENT

## Set constantes


In [2]:
PATH = 'data/htx_future'
TRADING_PAIRS = [
    'BTC-USDT',
    'ETH-USDT',
    'SOL-USDT',
    'XRP-USDT',
]
OUT_PUT_PATH = 'data/output'
DEPTH = 20

if not os.path.exists(OUT_PUT_PATH):
    os.makedirs(OUT_PUT_PATH)

## Define data preparing methods


In [3]:
def parse_trades(file_path):
    # Create an empty numpy array to store trade events with predefined dtype
    trades = []

    # Open the trade data file (assumed to be JSONL format, i.e. one JSON object per line)
    with open(file_path, 'r') as f:
        for line in f:
            try:
                entry = json.loads(line)
                tick = entry.get("tick")

                local_ts = int((entry.get("ts") + 0.5) * 1_000_000)
                data = tick.get("data")

                for trade in data:
                    exch_ts = int(trade.get("ts")) * 1_000_000
                    direction = BUY_EVENT if trade.get(
                        "direction") == "buy" else SELL_EVENT
                    px = float(trade.get("price", 0))
                    qty = float(trade.get("quantity", 0))

                    trades.append((
                        direction | TRADE_EVENT | EXCH_EVENT | LOCAL_EVENT,
                        exch_ts,
                        local_ts,
                        px,
                        qty,
                        0,                # order_id
                        0,                # ival
                        0.0               # faval
                    ))

            except:
                # Skip lines that are not valid JSON or do not match the expected structure
                continue

    print(trades)
    trades.sort(key=lambda x: (x[1], x[2]))
    return np.array(trades, dtype=event_dtype)

In [4]:
def parse_orderbook(file_path):
    records = []

    with open(file_path, 'r') as f:
        for line in f:
            try:
                entry = json.loads(line)
                tick = entry.get("tick")

                exch_ts = int(tick.get("ts")) * 1_000_000
                local_ts = int((entry.get("ts") + 0.5) * 1_000_000)

                bids = tick.get("bids", [])
                asks = tick.get("asks", [])

                for p, a in bids:
                    records.append((
                        BUY_EVENT | DEPTH_EVENT | EXCH_EVENT | LOCAL_EVENT,
                        exch_ts,
                        local_ts,
                        float(p),
                        float(a),
                        0,                # order_id
                        0,                # ival
                        0.0               # faval
                    ))

                for p, a in asks:
                    records.append((
                        SELL_EVENT | DEPTH_EVENT | EXCH_EVENT | LOCAL_EVENT,
                        exch_ts,
                        local_ts,
                        float(p),
                        float(a),
                        0,                # order_id
                        0,                # ival
                        0.0               # faval
                    ))

            except Exception:
                continue

    records.sort(key=lambda x: (x[1], x[2]))

    return np.array(records, dtype=event_dtype)

### Process Data


In [5]:
# test scripts
orderbook_file_path = 'data/htx_future/ETH-USDT/orderbook_20250607.json'
trade_file_path = 'data/htx_future/ETH-USDT/trade_20250607.json'

orderbook = parse_orderbook(orderbook_file_path)
print(f"example orderbook: \n{orderbook[:3]}")

trades = parse_trades(trade_file_path)
print(f"example trades: \n{trades[:3]}")

example orderbook: 
[(3758096385, 1749351817021000000, 1749351817026500096, 2510.51, 4341., 0, 0, 0.)
 (3758096385, 1749351817021000000, 1749351817026500096, 2510.48,   11., 0, 0, 0.)
 (3758096385, 1749351817021000000, 1749351817026500096, 2510.41,  111., 0, 0, 0.)]
[(3758096386, 1749351814104000000, 1749351814108499968, 2510.48, 3.0, 0, 0, 0.0), (3758096386, 1749351814104000000, 1749351814108499968, 2510.48, 2.22, 0, 0, 0.0), (3758096386, 1749351814104000000, 1749351814108499968, 2510.48, 0.4, 0, 0, 0.0), (3758096386, 1749351814104000000, 1749351814108499968, 2510.5, 0.16, 0, 0, 0.0), (3758096386, 1749351814104000000, 1749351814108499968, 2510.57, 0.24, 0, 0, 0.0), (3758096386, 1749351814104000000, 1749351814108499968, 2510.57, 2.22, 0, 0, 0.0), (3758096386, 1749351814104000000, 1749351814108499968, 2510.6, 0.2, 0, 0, 0.0), (3758096386, 1749351814104000000, 1749351814108499968, 2510.64, 0.88, 0, 0, 0.0), (3758096386, 1749351814104000000, 1749351814108499968, 2510.75, 6.96, 0, 0, 0.0),